# Peng-Robinson Equation of State: Root Finding

This notebook demonstrates GMM and Flow Matching approaches for finding roots of the PR-EOS cubic equation.

**Authors:** Victor Alves and John R. Kitchin

## System Information

This section documents the computational environment used to run this notebook.

In [ ]:
import sys
import platform
import time
import psutil
import os

print("System Information")
print("=" * 70)
print(f"Python version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")
print(f"CPU count: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count(logical=True)} logical")
print(f"Total memory: {psutil.virtual_memory().total / (1024**3):.2f} GB")
print("=" * 70)

# Key package versions
print("\nKey Library Versions:")
print("-" * 70)
packages = [
    ('numpy', 'np'),
    ('scipy', 'scipy'),
    ('matplotlib', 'matplotlib'),
    ('sklearn', 'sklearn'),
    ('torch', 'torch'),
    ('gmr', 'gmr'),
]

for pkg_name, import_name in packages:
    try:
        mod = __import__(pkg_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"{pkg_name:<20} {version}")
    except ImportError:
        print(f"{pkg_name:<20} NOT INSTALLED")

print("-" * 70)

# Initialize timing and memory tracking
_start_time = time.time()
_process = psutil.Process(os.getpid())
_start_memory_mb = _process.memory_info().rss / (1024 * 1024)

print(f"\nNotebook execution started: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_start_time))}")
print(f"Initial memory usage: {_start_memory_mb:.2f} MB")
print("=" * 70)

## System Information

This section documents the computational environment used to run this notebook.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
import os

# Force CPU
os.environ['JAX_PLATFORMS'] = 'cpu'

import torch

# Import reusable utilities
from generative_optimization import (
    generate_samples,
    best_gmm,
    ConditionalFlowMatching
)

# Figure settings for publication
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['font.size'] = 11
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['legend.fontsize'] = 10

warnings.filterwarnings('ignore')

# Set seeds
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete")

---
## Peng-Robinson Equation of State: Root Finding

The Peng-Robinson equation of state in compressibility factor form is a cubic equation:

$$Z^3 - (1-B)Z^2 + (A-3B^2-2B)Z - (AB-B^2-B^3) = 0$$

Where:
- $A = \frac{aP}{R^2T^2}$, $B = \frac{bP}{RT}$
- $a = 0.45724 \frac{R^2T_c^2}{P_c} \alpha(T)$, $b = 0.07780 \frac{RT_c}{P_c}$
- $\alpha(T) = [1 + \kappa(1-\sqrt{T/T_c})]^2$
- $\kappa = 0.37464 + 1.54226\omega - 0.26992\omega^2$

We use **propane** as the test substance with critical properties from NIST.

In [ ]:
# Peng-Robinson EOS for propane
# Critical properties (NIST)
Tc = 369.83  # K - Critical temperature
Pc = 42.48e5  # Pa (42.48 bar) - Critical pressure
omega = 0.1523  # Acentric factor
R = 8.314  # J/(mol*K) - Gas constant

# PR-EOS parameters
kappa = 0.37464 + 1.54226*omega - 0.26992*omega**2

def pr_eos_coefficients(T, P):
    """Calculate A, B and cubic coefficients for PR-EOS."""
    Tr = T / Tc
    alpha = (1 + kappa * (1 - np.sqrt(Tr)))**2
    
    a = 0.45724 * R**2 * Tc**2 / Pc * alpha
    b = 0.07780 * R * Tc / Pc
    
    A = a * P / (R**2 * T**2)
    B = b * P / (R * T)
    
    # Cubic: Z^3 + c2*Z^2 + c1*Z + c0 = 0
    c2 = -(1 - B)
    c1 = A - 3*B**2 - 2*B
    c0 = -(A*B - B**2 - B**3)
    
    return A, B, [1, c2, c1, c0]

def pr_eos_residual(Z, A, B):
    """Residual of PR-EOS cubic equation (should equal zero at roots)."""
    return Z**3 - (1-B)*Z**2 + (A - 3*B**2 - 2*B)*Z - (A*B - B**2 - B**3)

# Test conditions: T=300 K, P=10 bar (two-phase region for propane)
T_test, P_test = 300, 10e5
A, B, coeffs = pr_eos_coefficients(T_test, P_test)

# Analytical roots
roots_analytical = np.roots(coeffs)
real_roots = np.sort(roots_analytical[np.abs(roots_analytical.imag) < 1e-10].real)

print(f"Propane at T={T_test} K, P={P_test/1e5:.1f} bar")
print(f"A = {A:.6f}, B = {B:.6f}")
print(f"\nAnalytical Z roots:")
print(f"  Z_liquid   = {real_roots[0]:.6f}")
print(f"  Z_unstable = {real_roots[1]:.6f}")
print(f"  Z_vapor    = {real_roots[2]:.6f}")

### Train GMM for Root Finding

In [ ]:
# Build generative model for root finding
# Sample Z values and compute the cubic residual
# Use 2000 samples for sharper distributions (same as flow matching)
Z_samples = np.linspace(0.01, 0.99, 2000)
residual_samples = pr_eos_residual(Z_samples, A, B)

data_gmm = np.column_stack([Z_samples, residual_samples])

gmm_root, gmm_info = best_gmm(data_gmm, verbose=True)
print(f"\nBest GMM: {gmm_info['best_k']} components")

In [ ]:
# Find roots by conditioning on residual = 0
c_gmm = gmm_root.condition([1], [[0.0]])
Z_gmm_samples = c_gmm.sample(1000)[:, 0]

print("GMM root finding results (conditioning on residual=0):")
print(f"  Samples: {len(Z_gmm_samples)}")
print(f"  Mean: {Z_gmm_samples.mean():.4f}, Std: {Z_gmm_samples.std():.4f}")

### Train Flow Matching for Root Finding

In [ ]:
# Train Flow Matching models
# Uses the same 2000 samples as GMM for fair comparison

# Model 1: Generate Z conditioned on residual (for root finding)
x_data_fm = Z_samples.reshape(-1, 1)  # Z values to generate
c_data_fm = residual_samples.reshape(-1, 1)  # Residual to condition on

fm_root = ConditionalFlowMatching(x_dim=1, c_dim=1, hidden_dim=128, n_layers=4)
fm_history = fm_root.fit(x_data_fm, c_data_fm, epochs=2000, batch_size=128, verbose=True)

# Model 2: Generate residual conditioned on Z (for forward prediction / fitting)
fm_forward = ConditionalFlowMatching(x_dim=1, c_dim=1, hidden_dim=128, n_layers=4)
fm_forward_history = fm_forward.fit(c_data_fm, x_data_fm, epochs=2000, batch_size=128, verbose=True)
print("\nTrained FM forward model (Z -> residual) for visualization")

In [ ]:
# Find roots by conditioning on residual = 0
# Use more integration steps for more accurate sampling
Z_fm_samples = fm_root.sample(c_values=[[0.0]], n_samples=1000, n_steps=200)[:, 0]

print("Flow Matching root finding results (conditioning on residual=0):")
print(f"  Samples: {len(Z_fm_samples)}")
print(f"  Mean: {Z_fm_samples.mean():.4f}, Std: {Z_fm_samples.std():.4f}")

---
## Publication Figure: GMM vs Flow Matching Comparison

In [ ]:
# Create publication figure (2 panels stacked vertically)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))

# ============================================================
# Top: PR-EOS cubic polynomial and roots, with GMM and FM fits
# ============================================================
Z_plot = np.linspace(0.01, 0.99, 200)
residual_plot = pr_eos_residual(Z_plot, A, B)

# True PR-EOS cubic
ax1.plot(Z_plot, residual_plot, 'k-', lw=2.5, label='True PR-EOS')

# GMM prediction (forward: Z -> residual)
Z_plot_col = Z_plot.reshape(-1, 1)
residual_gmm_pred = gmm_root.predict([0], Z_plot_col)[:, 0]
ax1.plot(Z_plot, residual_gmm_pred, 'r--', lw=2, label='GMM fit')

# Flow Matching prediction (forward: Z -> residual)
# Use the fm_forward model which was trained to predict residual from Z
residual_fm_pred = fm_forward.predict(Z_plot_col, n_samples=50)
ax1.plot(Z_plot, residual_fm_pred, 'b:', lw=2, label='FM fit')

ax1.axhline(0, color='gray', ls='--', alpha=0.5)
ax1.plot(real_roots, [0, 0, 0], 'go', ms=10, zorder=5, label='Analytical roots')

ax1.set_xlabel('Compressibility Factor Z')
ax1.set_ylabel('Residual')
ax1.set_title(f'(a) PR-EOS: Propane at T={T_test} K, P={P_test/1e5:.0f} bar')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1)
ax1.set_ylim(-0.02, 0.04)

# ============================================================
# Bottom: Overlapping histograms for GMM and Flow Matching
# ============================================================
bins = np.linspace(0, 1, 80)

ax2.hist(Z_gmm_samples, bins=bins, alpha=0.5, color='red', 
         label='GMM', density=True)
ax2.hist(Z_fm_samples, bins=bins, alpha=0.5, color='blue', 
         label='Flow Matching', density=True)

for root in real_roots:
    ax2.axvline(root, color='black', ls='--', lw=2)
ax2.axvline(real_roots[0], color='black', ls='--', lw=2, label='Analytical')

ax2.set_xlabel('Compressibility Factor Z')
ax2.set_ylabel('Density')
ax2.set_title('(b) Root Finding: GMM vs Flow Matching')
ax2.legend(loc='upper center', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 1)

plt.tight_layout()
plt.savefig('preos_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved as 'preos_comparison.png'")

### Quantitative Comparison

In [ ]:
from generative_optimization import cluster_stats

print("Root Finding Comparison: Propane PR-EOS")
print(f"Conditions: T = {T_test} K, P = {P_test/1e5:.0f} bar")
print("=" * 70)

print(f"\nAnalytical roots:")
print(f"  Z_liquid   = {real_roots[0]:.6f}")
print(f"  Z_unstable = {real_roots[1]:.6f}")
print(f"  Z_vapor    = {real_roots[2]:.6f}")

print(f"\nGMM clusters (conditioned on residual=0):")
cluster_stats(Z_gmm_samples)

print(f"\nFlow Matching clusters (conditioned on residual=0):")
cluster_stats(Z_fm_samples)

# Verify residuals at sampled roots
residual_gmm = pr_eos_residual(Z_gmm_samples, A, B)
residual_fm = pr_eos_residual(Z_fm_samples, A, B)

print(f"\nResidual verification:")
print(f"  GMM:  mean|residual| = {np.mean(np.abs(residual_gmm)):.6f}")
print(f"  FM:   mean|residual| = {np.mean(np.abs(residual_fm)):.6f}")

### Physical Interpretation

In [ ]:
# Compute molar volumes from Z values
def Z_to_V(Z, T, P):
    """Convert compressibility factor to molar volume."""
    return Z * R * T / P

print("Physical interpretation of roots:")
print("=" * 50)
for i, (root, label) in enumerate(zip(real_roots, ['Liquid', 'Unstable', 'Vapor'])):
    V = Z_to_V(root, T_test, P_test)
    rho = 44.1 / (V * 1000)  # kg/m³ (propane MW = 44.1 g/mol)
    print(f"\n{label} phase:")
    print(f"  Z = {root:.6f}")
    print(f"  V = {V*1e6:.2f} cm³/mol")
    print(f"  ρ = {rho:.2f} kg/m³")

In [ ]:
# Final timing and memory report
_end_time = time.time()
_end_memory_mb = _process.memory_info().rss / (1024 * 1024)
_elapsed_seconds = _end_time - _start_time

print("\n" + "=" * 70)
print("NOTEBOOK EXECUTION SUMMARY")
print("=" * 70)
print(f"Start time:      {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_start_time))}")
print(f"End time:        {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_end_time))}")
print(f"Elapsed time:    {_elapsed_seconds:.2f} seconds ({_elapsed_seconds/60:.2f} minutes)")
print("-" * 70)
print(f"Initial memory:  {_start_memory_mb:.2f} MB")
print(f"Final memory:    {_end_memory_mb:.2f} MB")
print(f"Memory change:   {_end_memory_mb - _start_memory_mb:+.2f} MB")
print("=" * 70)